In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Mount Google Drive
# ════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Extract project
# ════════════════════════════════════════════════════════════════

import os, shutil

# Extract
!tar -xzf /content/drive/MyDrive/face-tracker-submission.tar.gz -C /content/

# Fix nested structure
src = '/content/face-tracker-submission/face-tracker'
dst = '/content/face-tracker'

if os.path.exists(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.move(src, dst)

# Move into correct directory
%cd /content/face-tracker

# Verify
!find . -maxdepth 2 -name "*.py" | head -30

print("✓ Extraction complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Drive symlinks (skips __pycache__ and subdirs)
# ════════════════════════════════════════════════════════════════
import os, shutil

ROOT    = '/content/face-tracker'
PROJECT = '/content/drive/MyDrive/face-tracker-project'

for folder in ['logs', 'models', 'insightface_cache', 'outputs']:
    os.makedirs(f'{PROJECT}/{folder}', exist_ok=True)

models_link = f'{ROOT}/models'
if os.path.islink(models_link):
    pass
elif os.path.isdir(models_link):
    for f in os.listdir(models_link):
        src = f'{models_link}/{f}'
        if os.path.isfile(src):
            shutil.copy2(src, f'{PROJECT}/models/{f}')
    shutil.rmtree(models_link)
    os.symlink(f'{PROJECT}/models', models_link)
else:
    os.symlink(f'{PROJECT}/models', models_link)


logs_link = f'{ROOT}/logs'
if not os.path.exists(logs_link) and not os.path.islink(logs_link):
    os.makedirs(f'{PROJECT}/logs', exist_ok=True)
    os.symlink(f'{PROJECT}/logs', logs_link)

db_pkg = f'{ROOT}/database'
if os.path.islink(db_pkg):
    target = os.readlink(db_pkg)
    os.unlink(db_pkg)
    os.makedirs(db_pkg, exist_ok=True)
    for bak, dst in [('/tmp/db_manager_bak.py', f'{db_pkg}/database_manager.py'),
                     ('/tmp/db_init_bak.py',    f'{db_pkg}/__init__.py')]:
        if os.path.exists(bak) and not os.path.exists(dst):
            shutil.copy2(bak, dst)

if os.path.exists(f'{db_pkg}/database_manager.py'):
    shutil.copy2(f'{db_pkg}/database_manager.py', '/tmp/db_manager_bak.py')
if os.path.exists(f'{db_pkg}/__init__.py'):
    shutil.copy2(f'{db_pkg}/__init__.py', '/tmp/db_init_bak.py')

insightface_local = '/root/.insightface'
insightface_drive = f'{PROJECT}/insightface_cache'
if os.path.exists(insightface_local) and not os.path.islink(insightface_local):
    shutil.copytree(insightface_local, insightface_drive, dirs_exist_ok=True)
    shutil.rmtree(insightface_local)
    os.symlink(insightface_drive, insightface_local)
elif not os.path.exists(insightface_local):
    os.symlink(insightface_drive, insightface_local)

print("Symlink / folder status:")
checks = [
    ('models/',       f'{ROOT}/models'),
    ('logs/',         f'{ROOT}/logs'),
    ('database/',     f'{ROOT}/database'),
    ('.insightface/', '/root/.insightface'),
]
for label, path in checks:
    if os.path.islink(path):
        print(f"  ✓ symlink  {label} → {os.readlink(path)}")
    elif os.path.isdir(path):
        py_files = [f for f in os.listdir(path) if f.endswith('.py')]
        print(f"  ✓ real dir {label} ({len(py_files)} .py files)")
    else:
        print(f"  ✗ MISSING  {label}")

import sys; sys.path.insert(0, '/content/face-tracker')
from database.database_manager import DatabaseManager
print("\n✓ database_manager import OK after symlink setup")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — Install packages
# ════════════════════════════════════════════════════════════════
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
!pip install ultralytics insightface onnxruntime-gpu opencv-python flask flask-cors numpy scipy Pillow -q
print("✓ All packages installed")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Download models (YOLO multi-source + InsightFace)
# ════════════════════════════════════════════════════════════════
import os, urllib.request, shutil

YOLO_PATH  = '/content/face-tracker/models/yolov8n-face.pt'
DRIVE_COPY = '/content/drive/MyDrive/face-tracker-project/models/yolov8n-face.pt'
MIN_SIZE   = 4_000_000

def valid(p): return os.path.exists(p) and os.path.getsize(p) >= MIN_SIZE

if valid(DRIVE_COPY) and not valid(YOLO_PATH):
    shutil.copy2(DRIVE_COPY, YOLO_PATH)
    print(f"✓ YOLO copied from Drive ({os.path.getsize(YOLO_PATH)/1e6:.1f} MB)")
elif valid(YOLO_PATH):
    print(f"✓ YOLO already present ({os.path.getsize(YOLO_PATH)/1e6:.1f} MB)")
else:
    SOURCES = [
        "https://github.com/akanametov/yolo-face/releases/download/v0.0.0/yolov8n-face.pt",
        "https://github.com/lindevs/yolov8-face/releases/latest/download/yolov8n-face-lindevs.pt",
        "https://github.com/SannketNikam/Face-Detection/raw/main/yolov8n-face.pt",
    ]
    downloaded = False
    for i, url in enumerate(SOURCES, 1):
        print(f"Trying source {i}/{len(SOURCES)}: {url.split('/')[2]} ...")
        try:
            tmp = YOLO_PATH + ".tmp"
            urllib.request.urlretrieve(url, tmp)
            if os.path.getsize(tmp) >= MIN_SIZE:
                os.rename(tmp, YOLO_PATH)
                shutil.copy2(YOLO_PATH, DRIVE_COPY)
                print(f"✓ Downloaded & cached ({os.path.getsize(YOLO_PATH)/1e6:.1f} MB)")
                downloaded = True; break
            else:
                os.remove(tmp); print(f"  ✗ Too small, trying next...")
        except Exception as e:
            print(f"  ✗ {e}")
    if not downloaded:
        print("Trying HuggingFace...")
        try:
            os.system("pip install huggingface_hub -q")
            from huggingface_hub import hf_hub_download
            p = hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection",
                                filename="model.pt",
                                local_dir="/content/face-tracker/models")
            os.rename(p, YOLO_PATH)
            shutil.copy2(YOLO_PATH, DRIVE_COPY)
            print(f"✓ HuggingFace download OK ({os.path.getsize(YOLO_PATH)/1e6:.1f} MB)")
        except Exception as e:
            print(f"✗ All sources failed: {e}")
            print("Manual: upload yolov8n-face.pt to /content/face-tracker/models/")

if valid(YOLO_PATH):
    from ultralytics import YOLO
    import numpy as np
    m = YOLO(YOLO_PATH)
    m(np.zeros((320,320,3), dtype=np.uint8), verbose=False)
    print(f"✓ YOLO verified | classes={m.names}")
else:
    print("✗ YOLO not ready")

iface_dir = '/root/.insightface/models/buffalo_l'
if os.path.exists(iface_dir) and len(os.listdir(iface_dir)) >= 3:
    print(f"✓ InsightFace cached ({len(os.listdir(iface_dir))} files)")
else:
    print("Downloading InsightFace buffalo_l (~300 MB)...")
    from insightface.app import FaceAnalysis
    app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection','recognition'])
    app.prepare(ctx_id=0, det_size=(640,640))
    print(f"✓ InsightFace ready ({len(os.listdir(iface_dir))} files)")

print(f"\nYOLO      : {'✓' if valid(YOLO_PATH) else '✗'}")
print(f"InsightFace: {'✓' if os.path.exists(iface_dir) else '✗'}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — System check
# ════════════════════════════════════════════════════════════════
import torch, cv2, os, sys
sys.path.insert(0, '/content/face-tracker')

print("=== System Check ===")
print(f"CUDA        : {torch.cuda.is_available()}")
print(f"GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > T4 GPU'}")
print(f"PyTorch     : {torch.__version__}")
print(f"OpenCV      : {cv2.__version__}")

try: import insightface; print("InsightFace : ✓")
except: print("InsightFace : ✗  re-run Cell 4")
try: from ultralytics import YOLO; print("Ultralytics : ✓")
except: print("Ultralytics : ✗  re-run Cell 4")

yolo_path = '/content/face-tracker/models/yolov8n-face.pt'
if os.path.exists(yolo_path) and os.path.getsize(yolo_path) > 4_000_000:
    m = YOLO(yolo_path)
    print(f"YOLO model  : ✓ {m.names}  ({os.path.getsize(yolo_path)/1e6:.1f} MB)")
else:
    print("YOLO model  : ✗  run Cell 5 to download")

iface_dir = '/root/.insightface/models/buffalo_l'
if os.path.exists(iface_dir) and len(os.listdir(iface_dir)) >= 3:
    print(f"InsightFace : ✓ buffalo_l ({len(os.listdir(iface_dir))} files)")
else:
    print("InsightFace : ✗  run Cell 5 to download")

video_folder = '/content/drive/MyDrive/Video Datasets'
if os.path.exists(video_folder):
    vids = sorted(f for f in os.listdir(video_folder) if f.endswith('.mp4'))
    print(f"\n✓ {len(vids)} videos found")
    for v in vids:
        print(f"   {v}  ({os.path.getsize(f'{video_folder}/{v}')/1e6:.1f} MB)")
else:
    print("✗ Video Datasets folder not found")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — Save config
# ════════════════════════════════════════════════════════════════
import json, shutil

config_path  = '/content/face-tracker/config/config.json'
drive_config = '/content/drive/MyDrive/face-tracker-project/config.json'

with open(config_path) as f:
    config = json.load(f)

config['detection']['device']               = 'cuda'
config['detection']['skip_frames']          = 1
config['detection']['confidence_threshold'] = 0.45
config['recognition']['ctx_id']             = 0
config['recognition']['model_name']         = 'buffalo_l'
config['recognition']['embedding_similarity_threshold'] = 0.38
config['recognition']['min_face_size']      = 35
config['tracking']['max_lost_frames']       = 30
config['tracking']['iou_threshold']         = 0.15
config['tracking']['min_hits']              = 2
config['frontend']['enabled']               = True
config['frontend']['host']                  = '0.0.0.0'
config['frontend']['port']                  = 5000
config['database']['type']                  = 'sqlite'
# NOTE: sqlite_path is set per-video in Cell 11 — this is just a placeholder
config['database']['sqlite_path']           = 'database/face_tracker.db'
config['logging']['log_file']               = 'logs/events.log'
config['logging']['image_store_base']       = 'logs'

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
shutil.copy(config_path, drive_config)
print("✓ Config saved")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — List available videos
# ════════════════════════════════════════════════════════════════
import os

video_folder = '/content/drive/MyDrive/Video Datasets'
videos = sorted(f for f in os.listdir(video_folder) if f.endswith('.mp4'))
print(f"Found {len(videos)} videos:\n")
for i, v in enumerate(videos):
    size = os.path.getsize(f'{video_folder}/{v}') / 1e6
    print(f"  [{i:2d}] {v}  ({size:.1f} MB)")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9 — Select videos
# ════════════════════════════════════════════════════════════════
# Edit this list to choose which videos to run
# Use the index numbers from Cell 9 output
# Example: VIDEOS_TO_PROCESS = [0, 1, 2]  ← runs first 3 videos
# Example: VIDEOS_TO_PROCESS = "all"      ← runs all videos

VIDEOS_TO_PROCESS = [7]

selected = videos if VIDEOS_TO_PROCESS == "all" else [videos[i] for i in VIDEOS_TO_PROCESS]
print(f"Will process {len(selected)} video(s):")
for v in selected:
    print(f"  → {v}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 10 — Run pipeline (no frontend)
# NOTE: Clear the existing outputs before running the pipeline
# ════════════════════════════════════════════════════════════════

import subprocess, os, shutil, json, time, sqlite3, sys

ROOT         = '/content/face-tracker'
video_folder = '/content/drive/MyDrive/Video Datasets'
PROJECT      = '/content/drive/MyDrive/face-tracker-project'
config_path  = f'{ROOT}/config/config.json'

def _ensure_db_package():
    """Guarantee database_manager.py exists in the package folder."""
    db_pkg = f'{ROOT}/database'

    if os.path.islink(db_pkg):
        os.unlink(db_pkg)
        os.makedirs(db_pkg, exist_ok=True)

    for bak, dst in [('/tmp/db_manager_bak.py', f'{db_pkg}/database_manager.py'),
                     ('/tmp/db_init_bak.py',    f'{db_pkg}/__init__.py')]:
        if not os.path.exists(dst) and os.path.exists(bak):
            shutil.copy2(bak, dst)
            print(f"  ↺ Restored {os.path.basename(dst)} from backup")

    if os.path.exists(f'{db_pkg}/database_manager.py'):
        shutil.copy2(f'{db_pkg}/database_manager.py', '/tmp/db_manager_bak.py')
    if os.path.exists(f'{db_pkg}/__init__.py'):
        shutil.copy2(f'{db_pkg}/__init__.py', '/tmp/db_init_bak.py')


def run_video(video_filename):
    video_path = f'{video_folder}/{video_filename}'
    video_stem = os.path.splitext(video_filename)[0]
    output_dir = f'{PROJECT}/outputs/{video_stem}'

    print(f"\n{'='*60}")
    print(f"Processing : {video_filename}")
    print(f"Output dir : {output_dir}")
    print(f"{'='*60}")

    # 1. Guarantee database Python package is intact
    _ensure_db_package()

    # 2. Create per-video output folders on Drive
    for folder in [f'{output_dir}/database',
                   f'{output_dir}/logs/entries',
                   f'{output_dir}/logs/exits']:
        os.makedirs(folder, exist_ok=True)

    # 3. Logs symlink — safe (no Python files in logs/)
    logs_link = f'{ROOT}/logs'
    if os.path.islink(logs_link):   os.unlink(logs_link)
    elif os.path.isdir(logs_link):  shutil.rmtree(logs_link)
    os.symlink(f'{output_dir}/logs', logs_link)

    # 4. Point config sqlite_path directly at per-video output folder (no symlink on database/ at all)
    with open(config_path) as f:
        config = json.load(f)
    config['video']['source']         = video_path
    config['video']['use_rtsp']       = False
    config['database']['sqlite_path'] = f'{output_dir}/database/face_tracker.db'
    config['logging']['log_file']     = 'logs/events.log'
    config['logging']['image_store_base'] = 'logs'
    config['frontend']['enabled']     = False

    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)

    # 5. Import check before launching subprocess
    chk = subprocess.run(
        ['python', '-c',
         'import sys; sys.path.insert(0,"/content/face-tracker"); '
         'from database.database_manager import DatabaseManager; print("ok")'],
        capture_output=True, text=True, cwd=ROOT)
    if 'ok' not in chk.stdout:
        print(f"✗ Import check failed:\n{chk.stderr}"); return None
    print("✓ Import check passed")

    # 6. Run pipeline
    start = time.time()
    proc = subprocess.Popen(
        ['python', 'main.py', '--source', video_path, '--no-preview', '--no-frontend'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=ROOT)
    for line in proc.stdout:
        print(line, end='')
    elapsed = time.time() - start

    # 7. Report
    db_path = f'{output_dir}/database/face_tracker.db'
    unique = 0
    report_lines = [f"VIDEO   : {video_filename}", f"RUNTIME : {elapsed:.1f}s"]
    if os.path.exists(db_path):
        conn = sqlite3.connect(db_path); conn.row_factory = sqlite3.Row
        row = conn.execute("SELECT total_unique FROM visitor_summary WHERE id=1").fetchone()
        unique = row['total_unique'] if row else 0
        evts = {r['event_type']: r['c'] for r in conn.execute(
            "SELECT event_type, COUNT(*) as c FROM events GROUP BY event_type").fetchall()}
        faces = conn.execute(
            "SELECT face_id, first_seen, entry_count FROM faces ORDER BY first_seen").fetchall()
        conn.close()
        report_lines += [f"UNIQUE : {unique}",
                         f"ENTRIES: {evts.get('entry',0)}",
                         f"EXITS  : {evts.get('exit',0)}",
                         "\nFACES:"]
        for face in faces:
            report_lines.append(f"  {face['face_id']}  seen={face['first_seen']}  visits={face['entry_count']}")

    with open(f'{output_dir}/report.txt', 'w') as rf:
        rf.write("\n".join(report_lines))

    entries = sum(len(fs) for _,_,fs in os.walk(f'{output_dir}/logs/entries'))
    exits   = sum(len(fs) for _,_,fs in os.walk(f'{output_dir}/logs/exits'))
    print(f"\n✓ {video_filename}  unique={unique}  entries={entries}  exits={exits}  {elapsed:.1f}s")
    return {'video': video_filename, 'unique': unique,
            'entries': entries, 'exits': exits, 'time': elapsed}


# ── Run ───────────────────────────────────────────────────────────
results = []
for video in selected:
    r = run_video(video)
    if r: results.append(r)

print(f"\n{'='*65}")
print(f"{'Video':<45} {'Unique':>7} {'Entries':>8} {'Exits':>6} {'Time':>7}")
print(f"{'-'*65}")
for r in results:
    print(f"{r['video'][:44]:<45} {r['unique']:>7} {r['entries']:>8} {r['exits']:>6} {r['time']:>6.1f}s")
print(f"{'='*65}")

# ── Generate metrics reports ──────────────────────────────────────
import sys

for mod in list(sys.modules.keys()):
    if 'generate_metrics' in mod or (mod == 'tools' ):
        del sys.modules[mod]

sys.path.insert(0, ROOT)
from tools.generate_metrics import run as gen_metrics

print("\nGenerating metrics reports...")
for r in results:
    video_stem = os.path.splitext(r['video'])[0]
    output_dir = f'{PROJECT}/outputs/{video_stem}'
    try:
        gen_metrics(output_dir)
    except Exception as e:
        print(f"  ✗ Metrics failed for {r['video']}: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 11 — WITH FRONTEND (one video, live dashboard)
# NOTE: Clear the existing outputs before running the pipeline
# ════════════════════════════════════════════════════════════════

import subprocess, os, shutil, json, time, socket, urllib.request, sqlite3

video_folder = '/content/drive/MyDrive/Video Datasets'
PROJECT      = '/content/drive/MyDrive/face-tracker-project'
config_path  = '/content/face-tracker/config/config.json'

# ── Change this to the video you want to demo ─────────────────────
VIDEO_NAME = 'record_20250620_183903.mp4'


os.system('npm install -g localtunnel -q 2>/dev/null')

video_path = f'{video_folder}/{VIDEO_NAME}'
video_stem = os.path.splitext(VIDEO_NAME)[0]
output_dir = f'{PROJECT}/outputs/{video_stem}'

print(f"Processing : {VIDEO_NAME}")
print(f"Output dir : {output_dir}")

# Create isolated output folders
for folder in [f'{output_dir}/database', f'{output_dir}/logs/entries', f'{output_dir}/logs/exits']:
    os.makedirs(folder, exist_ok=True)

with open(config_path) as f:
    config = json.load(f)

config['video']['source']             = video_path
config['video']['use_rtsp']           = False
config['database']['sqlite_path']     = f'{output_dir}/database/face_tracker.db'
config['logging']['log_file']         = 'logs/events.log'
config['logging']['image_store_base'] = 'logs'
config['frontend']['enabled']         = True
config['frontend']['host']            = '0.0.0.0'
config['frontend']['port']            = 5000

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

logs_link = '/content/face-tracker/logs'
if os.path.islink(logs_link):  os.unlink(logs_link)
elif os.path.isdir(logs_link): shutil.rmtree(logs_link)
os.symlink(f'{output_dir}/logs', logs_link)

# Kill old processes
os.system("pkill -f 'main.py' 2>/dev/null")
os.system("pkill -f 'lt --port' 2>/dev/null")
time.sleep(2)

# Start pipeline
process = subprocess.Popen(
    ['python', 'main.py', '--source', video_path, '--no-preview'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd='/content/face-tracker'
)

# Wait for Flask
print("Waiting for Flask", end='')
for _ in range(30):
    time.sleep(1)
    print('.', end='', flush=True)
    try:
        s = socket.create_connection(('localhost', 5000), timeout=1)
        s.close()
        print(' ✓ Flask ready!')
        break
    except:
        continue

# Start localtunnel
tunnel = subprocess.Popen(
    ['lt', '--port', '5000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
time.sleep(3)
url = tunnel.stdout.readline().strip()

try:
    ip = urllib.request.urlopen(
        'https://ipv4.icanhazip.com', timeout=5
    ).read().decode().strip()
except:
    ip = "visit icanhazip.com"

print(f"\n{'─'*60}")
print(f"✓ Dashboard URL      → {url}")
print(f"  Password if asked  → {ip}")
print(f"  /api/stats         → live stats")
print(f"  /api/events        → entry/exit log")
print(f"  /api/faces         → registered faces")
print(f"  /api/visitors      → unique count")
print(f"{'─'*60}\nLive logs:\n")

# Stream logs
start = time.time()
for line in process.stdout:
    print(line, end='')
elapsed = time.time() - start
tunnel.terminate()

# Save report
db_path = f'{output_dir}/database/face_tracker.db'
unique = 0
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    row  = conn.execute("SELECT total_unique FROM visitor_summary WHERE id=1").fetchone()
    unique = row[0] if row else 0
    conn.close()

print(f"\n✓ Done | Unique visitors: {unique} | Output: {output_dir}")

# ── Generate metrics report ───────────────────────────────────────
import sys
sys.path.insert(0, '/content/face-tracker')
from tools.generate_metrics import run as gen_metrics
try:
    gen_metrics(output_dir)
except Exception as e:
    print(f"  ✗ Metrics failed: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 12 — Verify output folder structures
# ════════════════════════════════════════════════════════════════

import os

PROJECT    = '/content/drive/MyDrive/face-tracker-project'
video_stem = os.path.splitext(selected[0])[0]   # first processed video
output_dir = f'{PROJECT}/outputs/{video_stem}'

print(f"Checking output structure for: {video_stem}\n")

# Check every required path exists
checks = [
    f'{output_dir}/database/face_tracker.db',
    f'{output_dir}/logs/events.log',
    f'{output_dir}/logs/entries',
    f'{output_dir}/logs/exits',
    f'{output_dir}/report.txt',
    f'{output_dir}/model_metrics.json',        # ← new
    f'{output_dir}/metrics_report.png',
    f'{output_dir}/model_metrics_report.png',  # ← new
    f'{output_dir}/metrics_summary.txt',
]

for path in checks:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) if exists and os.path.isfile(path) else None
    status = "✓" if exists else "✗ MISSING"
    size_str = f"({size/1024:.1f} KB)" if size else ""
    print(f"  {status}  {path.replace(output_dir, '.')}  {size_str}")

# Check date folders exist inside entries and exits
print("\nDate folders:")
for log_type in ['entries', 'exits']:
    base = f'{output_dir}/logs/{log_type}'
    date_folders = os.listdir(base) if os.path.exists(base) else []
    for date_folder in sorted(date_folders):
        images = os.listdir(f'{base}/{date_folder}')
        print(f"  ✓  logs/{log_type}/{date_folder}/  — {len(images)} images")
        for img in sorted(images)[:3]:   # show first 3
            print(f"       {img}")
        if len(images) > 3:
            print(f"       ... and {len(images)-3} more")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 13 — Verify output folder structures
# ════════════════════════════════════════════════════════════════

import sqlite3

db_path = f'{output_dir}/database/face_tracker.db'
conn    = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

print("── FACES TABLE ─────────────────────────────────────")
faces = conn.execute(
    "SELECT face_id, first_seen, last_seen, entry_count FROM faces ORDER BY first_seen"
).fetchall()
for f in faces:
    print(f"  {f['face_id']}  first={f['first_seen']}  last={f['last_seen']}  visits={f['entry_count']}")

print(f"\n── EVENTS TABLE (last 20) ──────────────────────────")
events = conn.execute(
    "SELECT face_id, event_type, timestamp, image_path FROM events ORDER BY timestamp DESC LIMIT 20"
).fetchall()
for e in events:
    print(f"  [{e['event_type'].upper():<5}]  {e['face_id']}  {e['timestamp']}  {e['image_path']}")

print(f"\n── VISITOR SUMMARY ─────────────────────────────────")
row = conn.execute("SELECT * FROM visitor_summary").fetchone()
print(f"  Total unique visitors: {row['total_unique']}")
print(f"  Last updated:          {row['last_updated']}")

conn.close()

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 14 — Log verification
# ════════════════════════════════════════════════════════════════

log_path = f'{output_dir}/logs/events.log'

# Count each event type
from collections import Counter
counts = Counter()

with open(log_path) as f:
    for line in f:
        for event in ['FACE_ENTRY', 'FACE_EXIT', 'RECOGNITION',
                      'EMBEDDING', 'REGISTRATION', 'TRACKING', 'SYSTEM']:
            if event in line:
                counts[event] += 1
                break

print("── events.log coverage ─────────────────────────────")
required = ['FACE_ENTRY', 'FACE_EXIT', 'RECOGNITION',
            'EMBEDDING', 'REGISTRATION', 'TRACKING', 'SYSTEM']
for event in required:
    count  = counts.get(event, 0)
    status = "✓" if count > 0 else "✗ MISSING"
    print(f"  {status}  {event:<20} : {count} lines")

# Show a sample of each event type
print("\n── Sample log lines ────────────────────────────────")
shown = set()
with open(log_path) as f:
    for line in f:
        for event in required:
            if event in line and event not in shown:
                print(f"  {line.strip()}")
                shown.add(event)
        if len(shown) == len(required):
            break

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 15 — Show image gallery
# ════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import os

PROJECT = '/content/drive/MyDrive/face-tracker-project'
OUTPUTS_DIR = f'{PROJECT}/outputs'

def show_gallery(folder, title, max_images=30):
    images = []
    if not os.path.exists(folder):
        print(f"Directory missing: {folder}")
        return

    for root, dirs, files in os.walk(folder):
        for f in sorted(files):
            if f.endswith(('.jpg', '.jpeg', '.png')):
                images.append(os.path.join(root, f))

    images = images[:max_images]
    if not images:
        print(f"No images in {folder}")
        return

    cols = 5
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows*3))

    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)

    if rows == 1 and cols == 1:
        axes = [axes]
    elif hasattr(axes, 'flatten'):
        axes = getattr(axes, 'flatten')()

    for i, path in enumerate(images):
        img = mpimg.imread(path)
        axes[i].imshow(img)
        axes[i].set_title(Path(path).stem[:16], fontsize=7) # Show ID & Timestamp
        axes[i].axis('off')

    for i in range(len(images), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()
    print(f"  → {len(images)} images generated for {title}")


# ── Automatically find and display Galleries for ALL processed videos ──

if os.path.exists(OUTPUTS_DIR):
    video_folders = sorted([f for f in os.listdir(OUTPUTS_DIR)
                            if os.path.isdir(os.path.join(OUTPUTS_DIR, f))])

    if not video_folders:
        print("No processed video outputs found in your Drive yet!")
    else:
        for video_name in video_folders:
            print(f"\n{'═'*80}")
            print(f"📷  DISPLAYING RESULTS FOR: {video_name}")
            print(f"{'═'*80}")

            base = f'{OUTPUTS_DIR}/{video_name}/logs'

            # Show Entries
            show_gallery(f'{base}/entries', f'ENTRIES — {video_name}')

            # Show Exits
            show_gallery(f'{base}/exits', f'EXITS  — {video_name}')
else:
    print(f"Output directory does not exist: {OUTPUTS_DIR}")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 16 - CLEAN EVERYTHING (SAFE VERSION)
# ════════════════════════════════════════════════════════════════
import os, shutil

ROOT    = '/content/face-tracker'
PROJECT = '/content/drive/MyDrive/face-tracker-project'

print("🧹 Cleaning everything...\n")

# ─────────────────────────────────────────────
# 1. DELETE ALL VIDEO OUTPUTS (MOST IMPORTANT)
# ─────────────────────────────────────────────
outputs_dir = f'{PROJECT}/outputs'

if os.path.exists(outputs_dir):
    shutil.rmtree(outputs_dir)
    os.makedirs(outputs_dir, exist_ok=True)
    print("✓ Deleted all video outputs")
else:
    print("• No outputs folder found")

# ─────────────────────────────────────────────
# 2. CLEAR LOGS (Drive)
# ─────────────────────────────────────────────
logs_dir = f'{PROJECT}/logs'

if os.path.exists(logs_dir):
    shutil.rmtree(logs_dir)
    os.makedirs(f'{logs_dir}/entries', exist_ok=True)
    os.makedirs(f'{logs_dir}/exits', exist_ok=True)
    print("✓ Cleared logs (entries + exits + events.log)")
else:
    print("• No logs folder found")

# ─────────────────────────────────────────────
# 3. DELETE DATABASE FILES (Drive)
# ─────────────────────────────────────────────
db_dir = f'{PROJECT}/database'

if os.path.exists(db_dir):
    for f in os.listdir(db_dir):
        if f.endswith(".db"):
            os.remove(f"{db_dir}/{f}")
    print("✓ Deleted database files")
else:
    print("• No database folder found")

# ─────────────────────────────────────────────
# 4. CLEAR LOCAL LOGS (SYMLINK SAFE)
# ─────────────────────────────────────────────
logs_link = f'{ROOT}/logs'

if os.path.islink(logs_link):
    target = os.readlink(logs_link)
    if os.path.exists(target):
        shutil.rmtree(target)
        os.makedirs(f'{target}/entries', exist_ok=True)
        os.makedirs(f'{target}/exits', exist_ok=True)
        print("✓ Cleared local logs (via symlink)")
elif os.path.isdir(logs_link):
    shutil.rmtree(logs_link)
    os.makedirs(f'{logs_link}/entries', exist_ok=True)
    os.makedirs(f'{logs_link}/exits', exist_ok=True)
    print("✓ Cleared local logs folder")

# ─────────────────────────────────────────────
# 5. DELETE LOCAL DATABASE FILES ONLY (SAFE)
# ─────────────────────────────────────────────
db_pkg = f'{ROOT}/database'

if os.path.isdir(db_pkg):
    for f in os.listdir(db_pkg):
        if f.endswith(".db"):
            os.remove(f"{db_pkg}/{f}")
    print("✓ Cleared local DB files (kept .py files safe)")

# ─────────────────────────────────────────────
# 6. CLEAR TEMP CACHE (OPTIONAL BUT GOOD)
# ─────────────────────────────────────────────
for tmp_file in ['/tmp/db_manager_bak.py', '/tmp/db_init_bak.py']:
    if os.path.exists(tmp_file):
        os.remove(tmp_file)

print("\n✅ CLEAN STATE READY")